# 01 - Load & Explore DAiSEE Dataset

Mục tiêu notebook này:
1. Load 3 file label (Train/Validation/Test)
2. Xem cấu trúc cột, phân bố class, tỉ lệ overlap giữa 4 dimension
3. Khám phá cấu trúc thư mục `DataSet/` để biết cách map ClipID -> path video thật
4. Sanity check: đếm số clip có label nhưng thiếu file, hoặc có file nhưng thiếu label

**Chưa extract landmark ở notebook này** — chỉ load và hiểu data trước.

In [1]:

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import cv2
import mediapipe as mp

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

print("Python:", sys.version)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn OK")

Matplotlib is building the font cache; this may take a moment.


Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
OpenCV: 5.0.0
MediaPipe: 1.0.0
Pandas: 3.0.5
Scikit-learn OK


In [3]:
import os
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)


## 1. Khai báo đường dẫn

Sửa `BASE_DIR` cho đúng path project trên máy bạn (Windows dùng raw string `r"..."` để tránh lỗi backslash).

In [5]:
# TODO: sửa path này cho đúng máy bạn
BASE_DIR = Path(r"C:\Users\KIET PC\Desktop\Focus_Guard")

DATASET_DIR = BASE_DIR / "archive(1)\DAiSEE\DataSet"
LABELS_DIR = BASE_DIR / "archive(1)\DAiSEE\Labels"

print("BASE_DIR exists:", BASE_DIR.exists())
print("DataSet exists:", DATASET_DIR.exists())
print("Labels exists:", LABELS_DIR.exists())


BASE_DIR exists: True
DataSet exists: True
Labels exists: True


<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
C:\Users\KIET PC\AppData\Local\Temp\ipykernel_6880\3241132539.py:4: SyntaxWarning: invalid escape sequence '\D'
  DATASET_DIR = BASE_DIR / "archive(1)\DAiSEE\DataSet"
C:\Users\KIET PC\AppData\Local\Temp\ipykernel_6880\3241132539.py:5: SyntaxWarning: invalid escape sequence '\D'
  LABELS_DIR = BASE_DIR / "archive(1)\DAiSEE\Labels"


## 2. Xem file bên trong Labels/

DAiSEE thường có 3 file: `TrainLabels.csv`, `ValidationLabels.csv`, `TestLabels.csv`.
Chạy cell dưới để confirm tên file chính xác (có thể khác chữ hoa/thường tuỳ bản tải về).

In [6]:
label_files = list(LABELS_DIR.glob("*.csv"))
print(f"Tìm thấy {len(label_files)} file CSV trong Labels/:")
for f in label_files:
    print(" -", f.name)


Tìm thấy 4 file CSV trong Labels/:
 - AllLabels.csv
 - TestLabels.csv
 - TrainLabels.csv
 - ValidationLabels.csv


In [7]:
# TODO: sửa lại tên file nếu khác so với mặc định bên dưới
train_labels = pd.read_csv(LABELS_DIR / "TrainLabels.csv")
val_labels   = pd.read_csv(LABELS_DIR / "ValidationLabels.csv")
test_labels  = pd.read_csv(LABELS_DIR / "TestLabels.csv")

print("Train:", train_labels.shape)
print("Val:  ", val_labels.shape)
print("Test: ", test_labels.shape)

train_labels.head()


Train: (5358, 5)
Val:   (1429, 5)
Test:  (1784, 5)


,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0


## 3. Kiểm tra tên cột

Cột thường thấy: `ClipID`, `Boredom`, `Engagement`, `Confusion`, `Frustration `
(lưu ý bản gốc DAiSEE hay có dấu space thừa ở cuối tên cột `Frustration ` — cần strip).

In [8]:
# Strip khoảng trắng thừa trong tên cột (lỗi phổ biến của DAiSEE gốc)
for df in [train_labels, val_labels, test_labels]:
    df.columns = df.columns.str.strip()

print(train_labels.columns.tolist())


['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration']


## 4. Phân bố class từng dimension

Xem distribution của từng dimension (0-3) trên tập Train — để biết mức độ imbalance.

In [9]:
dimensions = ["Boredom", "Engagement", "Confusion", "Frustration"]
dimensions = [d for d in dimensions if d in train_labels.columns]

for dim in dimensions:
    print(f"--- {dim} ---")
    print(train_labels[dim].value_counts().sort_index())
    print()


--- Boredom ---
Boredom
0    2433
1    1696
2    1073
3     156
Name: count, dtype: int64

--- Engagement ---
Engagement
0      34
1     213
2    2617
3    2494
Name: count, dtype: int64

--- Confusion ---
Confusion
0    3616
1    1245
2     431
3      66
Name: count, dtype: int64

--- Frustration ---
Frustration
0    4183
1     941
2     191
3      43
Name: count, dtype: int64



## 5. Kiểm tra overlap giữa các dimension

Đây là chỗ liên quan trực tiếp tới vấn đề "4 lớp overlap" đã bàn — xem thử bao nhiêu % sample có
đồng thời Boredom cao VÀ Confusion cao (mâu thuẫn/overlap thật sự tồn tại trong data, không phải giả định).

In [10]:
if "Boredom" in train_labels.columns and "Confusion" in train_labels.columns:
    overlap = train_labels[(train_labels["Boredom"] >= 2) & (train_labels["Confusion"] >= 2)]
    print(f"Số sample vừa Boredom>=2 vừa Confusion>=2: {len(overlap)} / {len(train_labels)} "
          f"({100*len(overlap)/len(train_labels):.1f}%)")


Số sample vừa Boredom>=2 vừa Confusion>=2: 192 / 5358 (3.6%)


## 6. Khám phá cấu trúc thư mục DataSet/

Chưa biết chắc DataSet chia theo Train/Validation/Test hay theo user ID trước.
List 2 cấp đầu để xem cấu trúc thật.

In [11]:
def list_tree(path, max_depth=2, prefix=""):
    if max_depth < 0:
        return
    entries = sorted(os.listdir(path))[:15]  # giới hạn 15 mục đầu để đỡ rối
    for entry in entries:
        full = Path(path) / entry
        print(prefix + entry + ("/" if full.is_dir() else ""))
        if full.is_dir() and max_depth > 0:
            list_tree(full, max_depth - 1, prefix + "    ")

list_tree(DATASET_DIR, max_depth=2)


Test/
    500044/
        5000441001/
        5000441002/
        5000441003/
        5000441005/
        5000441006/
        5000441007/
        5000441008/
        5000441009/
        5000441010/
        5000441012/
        5000441013/
        5000441014/
        5000441015/
        5000441016/
        5000441017/
    500067/
        5000671001/
        5000671002/
        5000671003/
        5000671004/
        5000671005/
        5000671006/
        5000671008/
        5000671009/
        5000671010/
        5000671011/
        5000671012/
        5000671013/
        5000671014/
        5000671015/
        5000671016/
    500095/
        5000951001/
        5000951002/
        5000951003/
        5000951004/
        5000951005/
        5000951006/
        5000951007/
        5000951008/
        5000951009/
        5000951010/
        5000951011/
        5000951012/
        5000951013/
        5000951015/
        5000951016/
    510009/
        5100091001/
        5100091003/
      

## 7. Map ClipID -> đường dẫn video thật

DataSet thường có cấu trúc: `DataSet/Train/<UserID>/<ClipID_folder_or_file>`.
Viết hàm search để tìm ra path thật cho 1 ClipID, dùng `rglob` (đệ quy) vì chưa chắc cấu trúc bao nhiêu cấp.

In [ ]:
def find_clip_path(clip_id, search_root):
    """Tìm file/folder tương ứng với clip_id trong search_root (đệ quy).
    clip_id trong CSV thường có đuôi .avi/.mp4 sẵn hoặc không, nên thử cả 2 kiểu match.
    """
    clip_stem = Path(clip_id).stem  # bỏ đuôi nếu có
    matches = list(Path(search_root).rglob(f"{clip_stem}*"))
    return matches

# Test thử với 1 ClipID đầu tiên trong Train
sample_clip_id = train_labels["ClipID"].iloc[0]
print("Đang tìm:", sample_clip_id)

found = find_clip_path(sample_clip_id, DATASET_DIR)
print(f"Tìm thấy {len(found)} kết quả khớp:")
for f in found[:5]:
    print(" -", f)


## 8. Sanity check toàn bộ Train

Chạy thử trên **20 clip đầu tiên** của Train trước (không chạy full 5000+ clip ở bước này —
việc này chỉ để confirm cách map đúng, chạy full sẽ để dành cho bước extract landmark sau).

In [ ]:
sample = train_labels.head(20)

missing = []
for clip_id in sample["ClipID"]:
    found = find_clip_path(clip_id, DATASET_DIR)
    if len(found) == 0:
        missing.append(clip_id)

print(f"Thiếu {len(missing)}/{len(sample)} clip trong 20 sample đầu.")
if missing:
    print("Danh sách thiếu:", missing)


## Next step

Nếu bước 7-8 map đúng (tìm ra file, không bị thiếu) → chuyển sang notebook `02_extract_landmarks.ipynb`
để chạy MediaPipe Face Mesh trên video, bắt đầu bằng thử nghiệm trên vài clip trước khi chạy full 3 split.